In [23]:
import pandas as pd
import numpy as np 
from scipy.sparse import hstack
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
import faiss
from sentence_transformers import SentenceTransformer
import os
from functools import lru_cache

In [24]:
df = pd.read_csv('../data/final_data/df_web.csv')
df

,Unnamed: 0,title,series,author,rating,description,language,genres,bookFormat,pages,publisher,publishDate,firstPublishDate,awards,coverImg
0,0,The Hunger Games,The Hunger Games #1,Suzanne Collins,4.33,WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...,en,"['adventure', 'dystopia', 'fantasy']",Hardcover,374,Scholastic Press,2008-09-14 00:00:00,False,True,https://i.gr-assets.com/images/S/compressed.ph...
1,1,Harry Potter and the Order of the Phoenix,Harry Potter #5,"J.K. Rowling, Mary GrandPré (Illustrator)",4.50,There is a door at the end of a silent corrido...,en,"['adventure', 'childrens', 'classics']",Paperback,870,Scholastic Inc.,2004-09-28 00:00:00,True,True,https://i.gr-assets.com/images/S/compressed.ph...
2,2,To Kill a Mockingbird,To Kill a Mockingbird,Harper Lee,4.28,The unforgettable novel of a childhood in a sl...,en,"['classics', 'fiction', 'historical']",Paperback,324,Harper Perennial Modern Classics,2006-05-23 00:00:00,True,True,https://i.gr-assets.com/images/S/compressed.ph...
3,3,Pride and Prejudice,Standalone Novel,"Jane Austen, Anna Quindlen (Introduction)",4.26,Alternate cover edition of ISBN 9780679783268S...,en,"['classics', 'fiction', 'historical']",Paperback,279,Modern Library,2000-10-10 00:00:00,True,False,https://i.gr-assets.com/images/S/compressed.ph...
4,4,Twilight,The Twilight Saga #1,Stephenie Meyer,3.60,About three things I was absolutely positive.\...,en,"['fantasy', 'fiction', 'paranormal']",Paperback,501,"Little, Brown and Company",2006-09-06 00:00:00,True,True,https://i.gr-assets.com/images/S/compressed.ph...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57300,74779,Beasts & Behemoths (Dungeons & Dragons),Standalone Novel,"Jim Zub, Stacy King, Andrew Wheeler, Official ...",4.04,Study this guide and keep it close at hand--th...,und,['fiction'],Paperback,114,Ten Speed Press,2020-10-20 00:00:00,True,False,http://books.google.com/books/content?id=1toui...
57301,74780,Faculty of Dragon Riders,Standalone Novel,Dmitry Nazarov,4.04,I tamed the Black Dragon!So I thought until I ...,und,['fiction'],Paperback,401,Litres,2022-08-24 00:00:00,True,False,http://books.google.com/books/content?id=QSGFE...
57302,74786,Midnight Delivery Sex,Standalone Novel,Neneko Narazaki,4.00,With SNS card for collecting in the first edit...,de,['comics'],Paperback,29,Hayabusa,2021-05-04 00:00:00,True,False,http://books.google.com/books/content?id=s_8_E...
57303,74787,Monster Girl: 2,Standalone Novel,Kazuki Funatsu,4.00,"After their meeting, Yatsuki finds herself hav...",it,"['comics', 'isekai']",Paperback,216,BD editions,2020-05-01 00:00:00,True,False,http://books.google.com/books/content?id=_yjnE...


In [25]:
df = df.drop('Unnamed: 0', axis =1)

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57305 entries, 0 to 57304
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   title             57305 non-null  object 
 1   series            57305 non-null  object 
 2   author            57305 non-null  object 
 3   rating            57305 non-null  float64
 4   description       57305 non-null  object 
 5   language          57305 non-null  object 
 6   genres            57305 non-null  object 
 7   bookFormat        57305 non-null  object 
 8   pages             57305 non-null  int64  
 9   publisher         57304 non-null  object 
 10  publishDate       57305 non-null  object 
 11  firstPublishDate  57305 non-null  bool   
 12  awards            57305 non-null  bool   
 13  coverImg          57305 non-null  object 
dtypes: bool(2), float64(1), int64(1), object(10)
memory usage: 5.4+ MB


We need to do Label Enconding with 'series', 'language' and 'bookformat'. 
The Booleans 'firstpublishdate' and 'awards' we change them to int. 
'Genres' we change it to multilabelbinarizer. 
We use the 'Author(s)' to make a 'rating' mean. 

In [27]:
df_model = df.copy()

cat_cols = ['series', 'language', 'bookFormat']
label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df_model[col] = df_model[col].astype(str).fillna('missing')
    df_model[col] = le.fit_transform(df_model[col])
    label_encoders[col] = le

df_model['firstPublishDate'] = df_model['firstPublishDate'].astype(int)
df_model['awards'] = df_model['awards'].astype(int)

author_avg = df_model.groupby('author')['rating'].mean()
df_model['author_rating'] = df_model['author'].map(author_avg)
df_model['author_rating'] = df_model['author_rating'].fillna(df_model['rating'].mean())

mlb = MultiLabelBinarizer()
df_model['genres'] = df_model['genres'].apply(lambda g: g if isinstance(g, list) else [])
genres_ohe = mlb.fit_transform(df_model['genres'])
df_genres = pd.DataFrame(genres_ohe, columns=mlb.classes_, index=df_model.index)

df_model = pd.concat([df_model, df_genres], axis=1)

In [28]:
df_model

,title,series,author,rating,description,language,genres,bookFormat,pages,publisher,publishDate,firstPublishDate,awards,coverImg,author_rating
0,The Hunger Games,16083,Suzanne Collins,4.33,WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...,14,[],35,374,Scholastic Press,2008-09-14 00:00:00,0,1,https://i.gr-assets.com/images/S/compressed.ph...,4.227143
1,Harry Potter and the Order of the Phoenix,6493,"J.K. Rowling, Mary GrandPré (Illustrator)",4.50,There is a door at the end of a silent corrido...,14,[],62,870,Scholastic Inc.,2004-09-28 00:00:00,1,1,https://i.gr-assets.com/images/S/compressed.ph...,4.570000
2,To Kill a Mockingbird,18386,Harper Lee,4.28,The unforgettable novel of a childhood in a sl...,14,[],62,324,Harper Perennial Modern Classics,2006-05-23 00:00:00,1,1,https://i.gr-assets.com/images/S/compressed.ph...,3.693333
3,Pride and Prejudice,13751,"Jane Austen, Anna Quindlen (Introduction)",4.26,Alternate cover edition of ISBN 9780679783268S...,14,[],62,279,Modern Library,2000-10-10 00:00:00,1,0,https://i.gr-assets.com/images/S/compressed.ph...,4.260000
4,Twilight,17776,Stephenie Meyer,3.60,About three things I was absolutely positive.\...,14,[],62,501,"Little, Brown and Company",2006-09-06 00:00:00,1,1,https://i.gr-assets.com/images/S/compressed.ph...,3.938750
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57300,Beasts & Behemoths (Dungeons & Dragons),13751,"Jim Zub, Stacy King, Andrew Wheeler, Official ...",4.04,Study this guide and keep it close at hand--th...,58,[],62,114,Ten Speed Press,2020-10-20 00:00:00,1,0,http://books.google.com/books/content?id=1toui...,4.040000
57301,Faculty of Dragon Riders,13751,Dmitry Nazarov,4.04,I tamed the Black Dragon!So I thought until I ...,58,[],62,401,Litres,2022-08-24 00:00:00,1,0,http://books.google.com/books/content?id=QSGFE...,4.040000
57302,Midnight Delivery Sex,13751,Neneko Narazaki,4.00,With SNS card for collecting in the first edit...,11,[],62,29,Hayabusa,2021-05-04 00:00:00,1,0,http://books.google.com/books/content?id=s_8_E...,4.000000
57303,Monster Girl: 2,13751,Kazuki Funatsu,4.00,"After their meeting, Yatsuki finds herself hav...",30,[],62,216,BD editions,2020-05-01 00:00:00,1,0,http://books.google.com/books/content?id=_yjnE...,4.000000


For the Reccomendation System we are going to use FAISS and all-MiniLM-L6-v2 to make embedding and that it combines the 'description' and 'genres' so when people make the searching, both columns will be use. 

In [29]:
# Prepare text for embedding

df['text_for_embedding'] = (
    df['title'].fillna('') + '. ' + 
    df['description'].fillna('') + '. Genres: ' + 
    df['genres'].apply(lambda g: ', '.join(g))
)

In [30]:
# Embedding & Index Persistence 

EMBED_FILE = 'book_embeddings.npy'
INDEX_FILE = 'book_index_hnsw.index'

In [31]:
# Load existing embeddings & index if present, else compute and save

if os.path.exists(EMBED_FILE) and os.path.exists(INDEX_FILE):
    print("[IMPROVEMENT] Loading cached embeddings and FAISS index...")
    embeddings = np.load(EMBED_FILE)
    model = SentenceTransformer('all-MiniLM-L6-v2')  # for query encoding
    index = faiss.read_index(INDEX_FILE)
else:
    print("[IMPROVEMENT] Computing embeddings and building FAISS index...")
    model = SentenceTransformer('all-MiniLM-L6-v2')
    emb_list = model.encode(df['text_for_embedding'].tolist(), show_progress_bar=True)
    embeddings = np.array(emb_list, dtype='float32')
    faiss.normalize_L2(embeddings)  # prepare for cosine similarity
    # Build HNSW index
    dim = embeddings.shape[1]
    index = faiss.IndexHNSWFlat(dim, 32)  # M=32 for good tradeoff
    index.hnsw.efConstruction = 200
    index.add(embeddings)
    # Persist to disk
    np.save(EMBED_FILE, embeddings)
    faiss.write_index(index, INDEX_FILE)

[IMPROVEMENT] Computing embeddings and building FAISS index...


Batches: 100%|██████████| 1791/1791 [41:07<00:00,  1.38s/it]


In [32]:
# Cache Query Embeddings

@lru_cache(maxsize=128)
def get_query_vec(text):
    vec = model.encode([text], convert_to_tensor=False)
    q = np.array(vec, dtype='float32')
    faiss.normalize_L2(q)
    return q

In [33]:
# Recommendation Function (uses improvements) 

def recommend_books(user_text, top_n=5):
    """Return top_n similar books for a text query."""
    q_vec = get_query_vec(user_text)  # cached
    distances, indices = index.search(q_vec, top_n)
    results = df.iloc[indices[0]].copy()
    results['score'] = distances[0]
    return results[['title','author','genres','rating','coverImg','description','score']]

In [37]:
# Example usage

if __name__ == '__main__':
    query = "I want to read a high fantasy story with epic battles, magic and dragons"
    recs = recommend_books(query, top_n=5)
    for _, row in recs.iterrows():
        print(f"{row['title']} by {row['author']} | score: {row['score']:.4f}")


Dragon Fairy Tales by Lizzie Stoddart | score: 0.7445
Historical Dictionary of Fantasy Literature by Allen Stroud | score: 0.7448
A Game of Thrones / A Clash of Kings by George R.R. Martin | score: 0.7904
The Story of Evil - An Epic Fantasy Saga by Tony Johnson | score: 0.8056
The Classic Collection of Fantasy. 45 novels, stories and poems. Illustrated by J. R. R. Tolkien, C. S. Lewis, William Morris, James Branch Cabell, Robert E. Howard, George MacDonald, Lewis Carroll, Charles Kingsley, Lord Dunsany | score: 0.8077
